In [ ]:
import synapse.ml.spark.aifunc as aifunc

In [ ]:
df = spark.table('mext.education_content_jp')
display(df)

In [ ]:
from pyspark.sql.functions import col

col_map = {
    '教材_キーワード': 'col_keywords',
    '教材_名称': 'col_name',
    '教材_形式': 'col_format',
    '教材_対象者': 'col_target_audience',
    '教材_教科等': 'col_subject',
    '教材_対象学年': 'col_target_grade',
    '教材_コンテンツ形式': 'col_content_type',
    '教材_価格_区分': 'col_price_category',
    '教材_状態': 'col_status',
    '教材_公開者': 'col_publisher',
}

ascii_df = df
for jp, en in col_map.items():
    ascii_df = ascii_df.withColumnRenamed(jp, en)

display(ascii_df)

In [ ]:
# ai.translate drops previous translated columns when chained.
# Workaround: translate each column independently, collect to pandas, join.
import pandas as pd

# Get the base data as pandas (only 887 rows)
base_pdf = ascii_df.toPandas()

translate_cols = [
    ('col_name', 'material_name'),
    ('col_keywords', 'material_keywords'),
    ('col_format', 'material_format'),
    ('col_target_audience', 'material_target_audience'),
    ('col_subject', 'material_subject'),
    ('col_target_grade', 'material_target_grade'),
    ('col_content_type', 'material_content_type'),
    ('col_price_category', 'material_price_category'),
    ('col_status', 'material_status'),
    ('col_publisher', 'material_publisher'),
]

for i, (input_col, output_col) in enumerate(translate_cols):
    print(f'Translating {i+1}/10: {input_col} -> {output_col}...')
    translated = ascii_df.ai.translate(to_lang='en', input_col=input_col, output_col=output_col)
    # Collect just the translated column
    t_pdf = translated.select(output_col).toPandas()
    base_pdf[output_col] = t_pdf[output_col]
    print(f'  Done - {base_pdf[output_col].notna().sum()} non-null values')

print(f'\nAll translations complete. Columns: {list(base_pdf.columns)}')
base_pdf.head()

In [ ]:
# Rename non-translated columns and build final table
rename_map = {
    '教材_ID': 'material_id',
    '教材_言語': 'material_language',
    '教材_分野_科目': 'material_field',
    '教材_ＵＲＬ': 'material_url',
    '教材_ライセンス': 'material_license',
}
base_pdf = base_pdf.rename(columns=rename_map)

# Select final columns in order
final_cols = [
    'material_id', 'material_name', 'material_language', 'material_keywords',
    'material_format', 'material_target_audience', 'material_subject', 'material_field',
    'material_target_grade', 'material_content_type', 'material_url',
    'material_price_category', 'material_license', 'material_status', 'material_publisher'
]
final_pdf = base_pdf[final_cols]

# Convert back to Spark and write
final_df = spark.createDataFrame(final_pdf)
final_df.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('mext.education_content_en')

result = spark.table('mext.education_content_en')
print(f'education_content_en: {result.count()} rows, {len(result.columns)} columns')
display(result)